# Oracle Construction — Arithmetic Oracle — Qiskit Notebook

**Primitive:** Arithmetic (modular-condition) phase oracle marking every basis state `x`
that satisfies `x mod MODULUS == REMAINDER`. Unlike the earlier single-target phase oracle,
this oracle marks **multiple** states — the standard technique when the marked set is
defined by an arithmetic predicate rather than a single fixed pattern: decompose the
predicate into its list of satisfying values, and apply one single-target phase-flip
sub-oracle per value (each sub-oracle touches a disjoint basis state, so they compose with
no interference, regardless of order).

## SETUP (must match the CUDA-Q notebook exactly)
| Parameter | Value |
|---|---|
| N_QUBITS | 3 |
| MODULUS | 3 |
| REMAINDER | 0 |
| MARKED_SET | `{x in 0..7 : x mod 3 == 0}` = `{0, 3, 6}` |
| SHOTS | 10000 |
| SEED | 42 |
| QUBIT ORDERING CONTRACT | qubit 0 = **rightmost** bit in every printed bitstring (Qiskit's native convention — no reversal needed in this notebook) |
| GLOBAL PHASE CONVENTION | normalize so the first amplitude with `abs(amp) > 1e-9`, taken in ascending bitstring order, is rotated to be real and positive |
| Basis ordering for matrices | `{|000>, |001>, ..., |111>}` ascending binary order |

## Theory recap

`O |x> = -|x>` if `x mod MODULUS == REMAINDER`, else `+|x>`. Each individual marked value
`m` is flipped using the same `X`-sandwich `H`-`MCX`-`H` `X`-undo trick as the single-target
phase oracle:

```
for m in MARKED_SET:
    for each qubit q where bit q of m is 0:
        X(q)
    H(q_{n-1})
    MCX(controls = q_0..q_{n-2}, target = q_{n-1})
    H(q_{n-1})
    for each qubit q where bit q of m is 0:
        X(q)
```

**Test harness:** `H` on every qubit first, so the sign flips are visible in the amplitudes.
As with the earlier phase oracles, this is a diagonal `±1` operator, so **measured
probabilities stay uniform `1/8`** — the arithmetic predicate is only visible in the
statevector's sign pattern (verified directly in Cell 6), not the histogram.

## Ordering-convention warning

Qiskit already indexes statevector amplitudes, `Operator` matrices, and measurement `counts`
keys with **qubit 0 as the least-significant (rightmost) bit** — no reversal is required in
this notebook. The CUDA-Q companion notebook indexes qubit 0 as the **most-significant
(leftmost) bit** internally and must explicitly reverse every bitstring.

**Run all cells top to bottom.**


In [1]:
# Install cell
!pip install -q qiskit qiskit-aer pylatexenc


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00


In [2]:
# Imports + constants
import numpy as np
from math import pi

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator
import matplotlib.pyplot as plt

N_QUBITS = 3
MODULUS = 3
REMAINDER = 0
MARKED_SET = [x for x in range(2**N_QUBITS) if x % MODULUS == REMAINDER]
SHOTS = 10000
SEED = 42
TOPIC_SLUG = "arith_oracle"

print("N_QUBITS   =", N_QUBITS)
print("MODULUS    =", MODULUS, " REMAINDER =", REMAINDER)
print("MARKED_SET =", MARKED_SET, [format(m, f'0{N_QUBITS}b') for m in MARKED_SET])
print("SHOTS      =", SHOTS)
print("SEED       =", SEED)


N_QUBITS   = 3
MODULUS    = 3  REMAINDER = 0
MARKED_SET = [0, 3, 6] ['000', '011', '110']
SHOTS      = 10000
SEED       = 42


In [ ]:
# Build the circuit gate-by-gate
qc = QuantumCircuit(N_QUBITS, name="ArithmeticOracleDemo")

for q in range(N_QUBITS):
    qc.h(q)  # test harness: uniform superposition so the sign pattern is visible

qc.barrier(label="oracle start")

for m in MARKED_SET:
    zero_bit_qubits = [q for q in range(N_QUBITS) if not ((m >> q) & 1)]
    for q in zero_bit_qubits:
        qc.x(q)  # remap m onto the all-ones pattern
    qc.h(N_QUBITS - 1)
    qc.mcx(list(range(N_QUBITS - 1)), N_QUBITS - 1)  # phase-flip via H-MCX-H sandwich
    qc.h(N_QUBITS - 1)
    for q in zero_bit_qubits:
        qc.x(q)  # undo

qc.barrier(label="oracle end")

print(qc)
print("\nMarked values processed, one sub-oracle each:", MARKED_SET)


In [ ]:
# Draw the circuit with matplotlib and save it
fig = qc.draw(output="mpl", style={"name": "iqp"})
fig.savefig(f"{TOPIC_SLUG}_qiskit_circuit.png", dpi=150, bbox_inches="tight")
fig


In [ ]:
# Explicit matrix representation, built two independent ways, cross-checked

def h_matrix_full(n, q):
    M = np.zeros((2**n, 2**n), dtype=complex)
    for idx in range(2**n):
        bit = (idx >> q) & 1
        idx0 = idx & ~(1 << q); idx1 = idx | (1 << q)
        if bit == 0:
            M[idx0, idx] += 1/np.sqrt(2); M[idx1, idx] += 1/np.sqrt(2)
        else:
            M[idx1, idx] += -1/np.sqrt(2); M[idx0, idx] += 1/np.sqrt(2)
    return M

def x_matrix_full(n, q):
    M = np.zeros((2**n, 2**n), dtype=complex)
    for idx in range(2**n):
        M[idx ^ (1 << q), idx] = 1
    return M

def mcx_matrix_full(n, controls, target):
    M = np.zeros((2**n, 2**n), dtype=complex)
    for idx in range(2**n):
        if all((idx >> c) & 1 for c in controls):
            M[idx ^ (1 << target), idx] = 1
        else:
            M[idx, idx] = 1
    return M

print("Ideal arithmetic-oracle matrix: diagonal, -1 at each marked index, +1 elsewhere:")
ideal_diag = np.array([-1.0 if x in MARKED_SET else 1.0 for x in range(2**N_QUBITS)], dtype=complex)
print(np.round(np.diag(ideal_diag), 4))

# (a) Hand-built full unitary (H layer + all sub-oracles)
U_hand = np.eye(2**N_QUBITS, dtype=complex)
for q in range(N_QUBITS):
    U_hand = h_matrix_full(N_QUBITS, q) @ U_hand
for m in MARKED_SET:
    zero_bit_qubits = [q for q in range(N_QUBITS) if not ((m >> q) & 1)]
    for q in zero_bit_qubits:
        U_hand = x_matrix_full(N_QUBITS, q) @ U_hand
    U_hand = h_matrix_full(N_QUBITS, N_QUBITS - 1) @ U_hand
    U_hand = mcx_matrix_full(N_QUBITS, list(range(N_QUBITS - 1)), N_QUBITS - 1) @ U_hand
    U_hand = h_matrix_full(N_QUBITS, N_QUBITS - 1) @ U_hand
    for q in zero_bit_qubits:
        U_hand = x_matrix_full(N_QUBITS, q) @ U_hand

# (b) Framework operator extraction
U_qiskit = Operator(qc).data
max_matrix_diff = np.max(np.abs(U_hand - U_qiskit))
print(f"\nMax |difference| between hand-built matrix and qiskit.quantum_info.Operator: {max_matrix_diff:.3e}")
assert max_matrix_diff < 1e-8
print("Full 8x8 unitary matrices MATCH.")

# isolate oracle-only sub-circuit and check against ideal diagonal
U_oracle_only = np.eye(2**N_QUBITS, dtype=complex)
for m in MARKED_SET:
    zero_bit_qubits = [q for q in range(N_QUBITS) if not ((m >> q) & 1)]
    for q in zero_bit_qubits:
        U_oracle_only = x_matrix_full(N_QUBITS, q) @ U_oracle_only
    U_oracle_only = h_matrix_full(N_QUBITS, N_QUBITS - 1) @ U_oracle_only
    U_oracle_only = mcx_matrix_full(N_QUBITS, list(range(N_QUBITS - 1)), N_QUBITS - 1) @ U_oracle_only
    U_oracle_only = h_matrix_full(N_QUBITS, N_QUBITS - 1) @ U_oracle_only
    for q in zero_bit_qubits:
        U_oracle_only = x_matrix_full(N_QUBITS, q) @ U_oracle_only

max_oracle_diff = np.max(np.abs(U_oracle_only - np.diag(ideal_diag)))
print(f"Max |difference| between oracle-only matrix and ideal diagonal: {max_oracle_diff:.3e}")
assert max_oracle_diff < 1e-8
print("Oracle-only matrix MATCHES the ideal arithmetic-predicate diagonal.")


In [ ]:
# Exact statevector: extract, apply global-phase normalization, print amplitudes + probabilities
qc_state_only = QuantumCircuit(N_QUBITS)
qc_state_only.compose(qc, inplace=True)

sv = Statevector.from_instruction(qc_state_only)
amps = sv.data.copy()

for idx in range(len(amps)):
    if abs(amps[idx]) > 1e-9:
        phase = np.angle(amps[idx])
        amps = amps * np.exp(-1j*phase)
        break

print("Qiskit convention: qubit 0 = rightmost bit -> NO reversal needed for these amplitudes.\n")
print(f"{'bitstring':10s} {'amplitude':22s} {'probability':12s}")
QISKIT_PROBS_EXACT = {}
for idx in range(2**N_QUBITS):
    bitstring = format(idx, f'0{N_QUBITS}b')
    amp = amps[idx]
    prob = abs(amp)**2
    QISKIT_PROBS_EXACT[bitstring] = prob
    marker = "  <-- x mod 3 == 0 (marked)" if idx in MARKED_SET else ""
    print(f"|{bitstring}>   {amp.real:+.6f}{amp.imag:+.6f}j    {prob:.6f}{marker}")

print("\nProbabilities sum to:", sum(QISKIT_PROBS_EXACT.values()))
print("\nNote: signs are correct only up to an overall global phase -- what matters is that")
print("marked states all share one sign and unmarked states all share the opposite sign.")


### Ordering-fix note (Qiskit)

**No bit reversal was applied in this notebook.** Qiskit's `Statevector`, `Operator`, and
measurement `counts` all natively index qubit 0 as the least-significant (rightmost) bit,
which already matches the QUBIT ORDERING CONTRACT.

Contrast this with the CUDA-Q notebook, where `cudaq.get_state()` / `cudaq.sample()` index
qubit 0 as the *most-significant* (leftmost) bit, requiring an explicit reversal step.


In [ ]:
# Measurement: add measurements, sample with the shared SEED/SHOTS, get counts -> probabilities
qc_meas = QuantumCircuit(N_QUBITS, N_QUBITS)
qc_meas.compose(qc, inplace=True)
qc_meas.measure(range(N_QUBITS), range(N_QUBITS))

sim = AerSimulator(seed_simulator=SEED)
qc_meas_t = transpile(qc_meas, sim)
job = sim.run(qc_meas_t, shots=SHOTS, seed_simulator=SEED)
counts = job.result().get_counts()

QISKIT_PROBS_SAMPLED = {format(idx, f'0{N_QUBITS}b'): 0.0 for idx in range(2**N_QUBITS)}
for bitstring, count in counts.items():
    QISKIT_PROBS_SAMPLED[bitstring] = count / SHOTS

print("Sampled probabilities (from", SHOTS, "shots, seed =", SEED, "):")
for bitstring in sorted(QISKIT_PROBS_SAMPLED):
    print(f"|{bitstring}>   {QISKIT_PROBS_SAMPLED[bitstring]:.4f}")


In [ ]:
# Histogram: matplotlib bar chart, saved to file
bitstrings_sorted = sorted(QISKIT_PROBS_EXACT.keys())
exact_vals = [QISKIT_PROBS_EXACT[b] for b in bitstrings_sorted]
sampled_vals = [QISKIT_PROBS_SAMPLED[b] for b in bitstrings_sorted]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(bitstrings_sorted))
width = 0.4
ax.bar(x - width/2, exact_vals, width, label="exact statevector", color="#1f77b4")
ax.bar(x + width/2, sampled_vals, width, label=f"sampled ({SHOTS} shots)", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels(bitstrings_sorted, rotation=0)
ax.set_xlabel("bitstring (qubit 0 = rightmost)")
ax.set_ylabel("probability")
ax.set_title(f"Arithmetic oracle demo (N={N_QUBITS}, x mod {MODULUS} == {REMAINDER}) — Qiskit")
ax.set_ylim(0, 0.25)
ax.legend()
fig.tight_layout()
fig.savefig(f"{TOPIC_SLUG}_qiskit_histogram.png", dpi=150, bbox_inches="tight")
fig


In [ ]:
# Resource counts
print("Circuit depth:", qc.depth())
print("Gate counts:", dict(qc.count_ops()))
print("Sub-oracles applied:", len(MARKED_SET))


### Convention notes (plain text summary)

- **Bit-order convention:** qubit 0 = rightmost character; Qiskit uses this natively, no
  reversal applied.
- **Global phase convention:** first amplitude (ascending bitstring order) with magnitude
  `> 1e-9` rotated to real & positive.
- **Basis ordering for matrices:** ascending binary order `|000> ... |111>`.
- **Why the histogram is flat:** an arithmetic phase oracle is still a diagonal `±1`
  matrix, so it cannot change any measurement probability -- only the sign pattern, which
  is verified directly in the statevector (Cell 6).


In [ ]:
# Final result
QISKIT_RESULT = {b: QISKIT_PROBS_EXACT[b] for b in sorted(QISKIT_PROBS_EXACT)}
print("QISKIT_RESULT =")
for b, p in QISKIT_RESULT.items():
    print(f"  '{b}': {p:.6f}")


In [ ]:
# Diff-check block (run AFTER also running the CUDA-Q notebook and pasting its CUDAQ_RESULT below)
HAND_COMPUTED_EXPECTED = {format(i, f'0{N_QUBITS}b'): 1/(2**N_QUBITS) for i in range(2**N_QUBITS)}
max_delta_vs_hand = max(abs(QISKIT_RESULT[b] - HAND_COMPUTED_EXPECTED[b]) for b in QISKIT_RESULT)
print("Max delta between QISKIT_RESULT and hand-computed expected (uniform 1/8):", max_delta_vs_hand)
assert max_delta_vs_hand < 1e-3

# CUDAQ_RESULT = { ... }  # <- paste the dict printed at the end of the CUDA-Q notebook here
# max_delta = max(abs(QISKIT_RESULT[b] - CUDAQ_RESULT[b]) for b in QISKIT_RESULT)
# print("Max probability delta between QISKIT_RESULT and CUDAQ_RESULT:", max_delta)
# assert max_delta < 1e-3, "Qiskit and CUDA-Q arithmetic-oracle results disagree beyond tolerance!"
